# MAIF Backtester — Quickstart

Open in Colab, edit the strategy cell below, then **Runtime → Run all**. You get back a scorecard.

No local install, no API keys. Uses Yahoo Finance for SPY data.

## 1 · Setup — clone the repo and install dependencies

Takes ~30 seconds. Run once per Colab session.

In [ ]:
!git clone -q https://github.com/robbiebusinessacc/maif-backtester.git
%cd maif-backtester
!pip install -q -r requirements.txt yfinance

## 2 · Write your strategy

Replace the logic inside `generate_signals`. Return `Signal.BUY`, `Signal.SELL`, or `Signal.HOLD` for each bar. The framework handles position sizing, execution, frictions, and reporting.

In [ ]:
import pandas as pd
from strategy.base import Strategy, Signal


class MyStrategy(Strategy):
    @property
    def name(self) -> str:
        return "My Strategy"

    def generate_signals(self, df: pd.DataFrame) -> pd.Series:
        signals = pd.Series(Signal.HOLD, index=df.index)

        # ── YOUR LOGIC HERE ──────────────────────────────
        # Example: SMA crossover (10 / 30)
        fast = df["Close"].rolling(10).mean()
        slow = df["Close"].rolling(30).mean()

        for i in range(1, len(df)):
            if fast.iloc[i] > slow.iloc[i] and fast.iloc[i - 1] <= slow.iloc[i - 1]:
                signals.iloc[i] = Signal.BUY
            elif fast.iloc[i] < slow.iloc[i] and fast.iloc[i - 1] >= slow.iloc[i - 1]:
                signals.iloc[i] = Signal.SELL
        # ─────────────────────────────────────────────────

        return signals

## 3 · Fetch data and run both engines

Runs the strategy through the bar engine (market orders at next open) and the event-driven engine (stop/limit/OCO intrabar). If the two disagree, the strategy is execution-sensitive — flagged on the scorecard.

In [ ]:
from datetime import date
from data_layer import DataLayer, YahooFinanceProvider
from backtester import BacktestConfig, Backtester, EventDrivenBacktester

dl = DataLayer()
dl.add_provider(YahooFinanceProvider())
df = dl.fetch("SPY", date(2022, 1, 1), date(2025, 1, 1))

config = BacktestConfig(
    initial_capital=100_000,
    commission_per_order=1.00,
    slippage_bps=2.0,
)

bar_result = Backtester(config).run(MyStrategy(), df)
event_result = EventDrivenBacktester(config).run(MyStrategy(), df)

print(f"{'Engine':<14} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Trades':>8}")
print("-" * 52)
for label, r in (("Bar", bar_result), ("Event-driven", event_result)):
    print(f"{label:<14} {r.total_return_pct:>9.1f}% {r.sharpe_ratio:>8.2f} {r.max_drawdown_pct:>7.1f}% {r.num_trades:>8d}")
print("-" * 52)
print(f"Benchmark (SPY buy & hold): {bar_result.benchmark_return_pct:+.1f}%")
divergence = abs(bar_result.total_return_pct - event_result.total_return_pct)
print(f"Engine divergence: {divergence:.2f} pp  {'(execution-sensitive)' if divergence > 1 else '(robust)'}")

## 4 · Generate the full scorecard

Takes ~30–60 s. Runs Monte Carlo stress tests (GBM, GAN regimes, noise injection — ~280 backtests), compares engines, and produces the four-page PNG report with letter grades.

In [ ]:
import os
from backtester import generate_scorecard
from IPython.display import Image, display

generate_scorecard(
    strategy=MyStrategy(),
    df=df,
    config=config,
    output_path="my_scorecard.png",
    event_driven_results={"result": event_result},
)

for suffix in ("_scorecard", "_bartest", "_eventdriven", "_montecarlo"):
    path = f"my_scorecard{suffix}.png"
    if os.path.exists(path):
        print(path)
        display(Image(path))

## That's it

Read the grades honestly. A strategy that passes Monte Carlo stress, survives the GAN crash regime, and agrees across both execution engines is worth trading. One that doesn't isn't — no matter how pretty the historical equity curve looks.

- Full site and all built-in scorecards: **https://maif.robbiew.dev**
- Source: **https://github.com/robbiebusinessacc/maif-backtester**